In [1]:
from PPairS.constants import data_path, results_path
import pandas as pd
from sklearn.metrics import f1_score as f1
import torch as t

In [2]:
dataset_aspects = {
    'newsroom': ['coherence', 'fluency', 'informativeness', 'relevance'],
    'summeval': ['coherence', 'consistency', 'fluency', 'relevance'],
    'hanna': ['coherence', 'complexity', 'empathy', 'engagement', 'relevance', 'surprise'],
    'rocstories': ['consistency'],
    'mctaco': [None],
    'caters': [None],
}

model = "llama-3.1-70b"

In [3]:
def read_probes(probe_dataset):
    # load supervised and unsupervised probes
    PATH = "/gws/nopw/j04/ai4er/users/maiush/PPairS_results"
    probe_s = t.load(f"{PATH}/{probe_dataset}/{model}/probe_s.pt", weights_only=True)
    probe_u = t.load(f"{PATH}/{probe_dataset}/{model}/probe_u.pt", weights_only=True)
    return probe_s, probe_u


def read_data(dataset, aspect):
    label_path = f'{data_path}/{dataset}'
    if dataset != "rocstories": 
        label_path += "_pairwise_comparisons"
    label_path += ".jsonl"
    data = pd.read_json(label_path, orient='records', lines=True)

    # load contrast pair activations
    act_path = f"{results_path}/{dataset}/{model}/"
    if aspect is not None: act_path += f'{aspect}_'
    act_path += 'contrast'
    x1 = t.load(f"{act_path}_1.pt", weights_only=True).float()
    x2 = t.load(f"{act_path}_2.pt", weights_only=True).float()
    # centering
    x1 -= x1.mean(0)
    x2 -= x2.mean(0)
    # contrast pair differences
    x = x1 - x2
    # labels
    if dataset == 'rocstories' or dataset == 'mctaco':
        c = 'correct'
    elif dataset == 'caters':
        c = 'first'
    else:
        c = aspect
    y = t.tensor(data[c], dtype=int)
    # mask out ties
    mask = y != -1
    x, y = x[mask], y[mask]
    return x, y

def evaluate(x, y, probe_s, probe_u):
    # supervised probes
    scores = []
    for s in probe_s:
        proj = x @ s
        p1 = (proj > 0).int() + 1
        p2 = (proj < 0).int() + 1
        p1 = f1(y, p1, labels=[1,2], average="weighted").item()
        p2 = f1(y, p2, labels=[1,2], average="weighted").item()
        scores.append(max(p1, p2))
    score_s = sum(scores) / len(scores)
    # unsupervised probes
    scores = []
    for u in probe_u:
        proj = x @ u
        p1 = (proj > 0).int() + 1
        p2 = (proj < 0).int() + 1
        p1 = f1(y, p1, labels=[1,2], average="weighted").item()
        p2 = f1(y, p2, labels=[1,2], average="weighted").item()
        scores.append(max(p1, p2))
    score_u = sum(scores) / len(scores)
    return score_s, score_u

In [5]:
all_results = pd.DataFrame(columns=["probe", "evaluate", "supervised", "unsupervised"])
for probe_dataset in dataset_aspects.keys():
    probe_s, probe_u = read_probes(probe_dataset)
    # evaluation datasets
    for dataset in dataset_aspects.keys():
        if probe_dataset == dataset: continue
        score_s, score_u = [], []
        for aspect in dataset_aspects[dataset]:
            x, y = read_data(dataset, aspect)
            _score_s, _score_u = evaluate(x, y, probe_s, probe_u)
            score_s.append(_score_s)
            score_u.append(_score_u)
        score_s = sum(score_s) / len(score_s)
        score_u = sum(score_u) / len(score_u)
        all_results.loc[len(all_results)] = [probe_dataset, dataset, score_s, score_u]

In [8]:
all_results

,probe,evaluate,supervised,unsupervised
0,newsroom,summeval,0.656334,0.763552
1,newsroom,hanna,0.668013,0.706185
2,newsroom,rocstories,0.871209,0.985435
3,newsroom,mctaco,0.792397,0.950961
4,newsroom,caters,0.747354,0.778421
5,summeval,newsroom,0.630924,0.772725
6,summeval,hanna,0.618735,0.705939
7,summeval,rocstories,0.777814,0.985368
8,summeval,mctaco,0.666833,0.949360
9,summeval,caters,0.621831,0.778321
